# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR^2) Exploration with `mlcroissant`

This notebook explores the FAIR^2 dataset ("Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution") using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. The dataset is described by a Croissant schema and contains rich tabular clinical oncology data.

### Dataset Source
- Dataset Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- [Publication (Frontiers)](https://sen.science/doi/10.71728/senscience.qs2f-h81p)


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and objects using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant Dataset
dataset = mlc.Dataset(croissant_url)

# Get top-level dataset metadata (as an object)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"\nLicense: {metadata.license}")
print(f"Published: {metadata.datePublished}\n")

## 2. Data Overview
List available record sets and their fields. All references use entity `@id`s as defined in the Croissant schema.

**Note:** In Croissant, `record sets` correspond to logical tables in the dataset. Each record set has a unique `@id`, and fields/columns are also referred to by `@id`.

In [ ]:
# List all available record sets and fields using their @id's
record_sets = list(dataset.record_sets)
if not record_sets:
    # fallback: try via metadata, if no explicit record_sets found
    record_sets = [rs for rs in getattr(metadata, 'recordSet', [])]  # Should be a list of RecordSet objects
    if not record_sets:
        raise ValueError("No record sets found in the dataset.")

print(f"Available record sets and their fields:\n")
rs_ids = []
for rs in record_sets:
    # If loaded via dataset.record_sets, rs is an mlc.RecordSet object
    rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else getattr(rs, '@id', None) or getattr(rs, 'id', None)
    if not rs_id:
        # Try as string
        rs_id = str(rs)
    rs_ids.append(rs_id)
    print(f"- RecordSet @id: {rs_id}")
    
    # Fetch fields for the record set
    if hasattr(rs, 'fields'):
        fields = rs.fields
    elif isinstance(rs, dict) and 'field' in rs:
        fields = rs['field']
    else:
        # Use dataset utility
        fields = []
        try:
            fields = [f for f in dataset.fields(record_set=rs_id)]
        except Exception:
            pass
    if fields:
        print("  - Fields:")
        for field in fields:
            field_id = field['@id'] if isinstance(field, dict) and '@id' in field else getattr(field, '@id', None)
            if not field_id:
                field_id = str(field)
            print(f"    - {field_id}")
    else:
        print("  - (No fields found)")
print(f"\nAll detected record sets: {rs_ids}")

## 3. Data Extraction
Load the actual tabular records from each record set using their `@id` field. Each DataFrame uses columns named by field `@id`s.

You can adjust the `record_set_ids` list to load only specific record sets as needed.

In [ ]:
# Use the full list of detected record set @id's
record_set_ids = rs_ids  # Obtained in the previous step

dataframes = {}
for rs_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {rs_id}")
    try:
        df = pd.DataFrame(dataset.records(record_set=rs_id))
        if not df.empty:
            dataframes[rs_id] = df
            print(f"  - Columns: {df.columns.tolist()}")
            print(df.head(2), "\n")
        else:
            print("  - No data.")
    except Exception as e:
        print(f"  - Error: {e}")

# Choose the first non-empty record set for further analysis
primary_rs_id = None
for rid, df in dataframes.items():
    if not df.empty:
        primary_rs_id = rid
        break

if not primary_rs_id:
    raise RuntimeError("No record set with non-empty records found.")

print(f"Selected primary record set for EDA: {primary_rs_id}")
print(f"Available columns (@id): {dataframes[primary_rs_id].columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
You can explore the primary record set by filtering, normalizing numeric fields, and grouping/categorizing by other key attributes. Field and group names are referenced using `@id`s.

**Tip:** To see all field @id's and some values, review the output above or use `.columns` and `.head()` on the DataFrame. Adjust `numeric_field` and `group_field` IDs below as appropriate for your analysis.

In [ ]:
# List fields with numeric-like data to select for EDA
df = dataframes[primary_rs_id]
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print(f"Numeric candidate fields (by @id): {numeric_candidates}")

# If no numeric dtype detected, try parsing all with pandas to find one
if not numeric_candidates:
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='ignore')
        except Exception:
            continue
    numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    print(f"After conversion, numeric candidate fields: {numeric_candidates}")

if not numeric_candidates:
    raise ValueError("No numeric field detected in the primary record set.")

# Select the first numeric field and group field for demonstration
numeric_field_id = numeric_candidates[0]
print(f"Using numeric field: {numeric_field_id}")

# Choose a group field by looking for a likely categorical variable
possible_group_fields = [col for col in df.columns if pd.api.types.is_object_dtype(df[col])]  # Simple guess
group_field_id = possible_group_fields[0] if possible_group_fields else None
if group_field_id:
    print(f"Grouping by field: {group_field_id}")
else:
    print("No categorical field found for grouping.")

###############################################################
# FILTERING: Filter for values greater than a threshold value  #
###############################################################

threshold = df[numeric_field_id].mean()  # Use mean as a threshold for demo
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records where '{numeric_field_id}' > {threshold:.2f} (mean): {len(filtered_df)} records")
print(filtered_df[[numeric_field_id]].head())

# Normalization (z-score)
filtered_df[f"{numeric_field_id}_normalized"] = (
    (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
)
print(f"\nNormalized '{numeric_field_id}':")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# GROUPING: by group_field_id, if available
if group_field_id and group_field_id in filtered_df:
    # Usually 'mean' only for numeric columns
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean of '{numeric_field_id}' by '{group_field_id}':")
    print(grouped.head())

## 5. Visualization
You can visualize the distribution of the selected numeric field and, if relevant, the grouping/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id], kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# If group_field_id is available, show boxplot/grouped plot
if group_field_id:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.xticks(rotation=30)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
- In this notebook, you learned how to access and explore a Croissant-described dataset using `mlcroissant`.
- The FAIR^2 dataset contains detailed clinicopathological data for secondary colorectal cancers in cancer survivors. 
- You loaded record sets using their `@id`, explored fields, performed basic filtering, normalization, and visualized numeric feature distributions, all referencing entities by `@id`.

**Next Steps:**
- Apply further processing or model-building as needed for clinical/biomedical data science!
- For more, see: https://github.com/mlcommons/croissant and the [mlcroissant documentation](https://mlcroissant.readthedocs.io/).
